<a href="https://colab.research.google.com/github/BRUTOCROSS/Argofloat-Data/blob/main/argo_dataset_2_years_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "argopy>=1.3.0" xarray pandas pyarrow netCDF4

In [ ]:
from argopy import DataFetcher as ArgoDataFetcher
import pandas as pd
import pathlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pathlib
out_dir = pathlib.Path("/content/drive/MyDrive/All_argo_data_2Years")
out_dir.mkdir(exist_ok=True)

In [ ]:
# --- Step 2: Import packages ---
import xarray as xr
from datetime import datetime, timedelta

# --- Step 4: Define region (Indian Ocean approx) ---
lon_min, lon_max = 20.0, 120.0     # longitude
lat_min, lat_max = -60.0, 30.0     # latitude
depth_min, depth_max = 0.0, 2000.0 # depth range

# --- Step 5: Define date range ---
start_date = datetime(2023, 1, 1)
end_date = datetime(2025, 7, 6)

# --- Step 6: Create Ifremer fetcher ---
fetcher = ArgoDataFetcher(src="erddap", server="https://erddap.ifremer.fr/erddap")

# --- Step 7: Loop through weekly chunks ---
current = start_date
all_datasets = []

while current < end_date:
    next_date = current + timedelta(days=7)  # weekly chunk
    if next_date > end_date:
        next_date = end_date

    # Define the box for this chunk
    box = [lon_min, lon_max, lat_min, lat_max, depth_min, depth_max,
           current.strftime("%Y-%m-%d"), next_date.strftime("%Y-%m-%d")]

    print(f"🌊 Fetching {box[6]} → {box[7]} ...")

    try:
        # Fetch data and save
        ds = fetcher.region(box).to_xarray()
        all_datasets.append(ds)

        # Save NetCDF per chunk
        filename_nc = f"argo_{current.strftime('%Y%m%d')}_{next_date.strftime('%Y%m%d')}.nc"
        ds.to_netcdf(out_dir / filename_nc)
        print(f"✅ Saved: {filename_nc}")

        # Save summary as parquet
        df = fetcher.region(box).to_dataframe()
        filename_parquet = f"argo_summary_{current.strftime('%Y%m%d')}_{next_date.strftime('%Y%m%d')}.parquet"
        df.to_parquet(out_dir / filename_parquet, index=False)
        print(f"📑 Summary saved: {filename_parquet}")

    except Exception as e:
        print(f"❌ Failed for {box[6]} → {box[7]}: {e}")

    # Move to next chunk
    current = next_date

print("✅ Finished fetching all chunks.")

# --- Step 8: Merge all datasets into one (optional) ---
if all_datasets:
    ds_all = xr.concat(all_datasets, dim="N_PROF")
    ds_all.to_netcdf(out_dir / "argo_indian_ocean_2023_2025.nc")
    print("🌍 Combined dataset saved:", out_dir / "argo_indian_ocean_2023_2025.nc")


🌊 Fetching 2023-01-01 → 2023-01-08 ...
❌ Failed for 2023-01-01 → 2023-01-08: 
🌊 Fetching 2023-01-08 → 2023-01-15 ...
Error: 502, message='Proxy Error', url='https://erddap.ifremer.fr/erddap/tabledap/ArgoFloats.nc?config_mission_number,cycle_number,data_mode,direction,latitude,longitude,platform_number,position_qc,pres,pres_adjusted,pres_adjusted_error,pres_adjusted_qc,pres_qc,psal,psal_adjusted,psal_adjusted_error,psal_adjusted_qc,psal_qc,temp,temp_adjusted,temp_adjusted_error,temp_adjusted_qc,temp_qc,time,time_qc,vertical_sampling_scheme&longitude%3E=20.0&longitude%3C=120.0&latitude%3E=-60.0&latitude%3C=30.0&pres%3E=0.0&pres%3C=2000.0&time%3E=1673136000.0&time%3C=1673740800.0&latitude!=NaN&longitude!=NaN&distinct()&orderBy(%22time,pres%22)'
❌ Failed for 2023-01-08 → 2023-01-15: https://erddap.ifremer.fr/erddap/tabledap/ArgoFloats.nc?config_mission_number,cycle_number,data_mode,direction,latitude,longitude,platform_number,position_qc,pres,pres_adjusted,pres_adjusted_error,pres_adjusted

KeyboardInterrupt: 

In [ ]:
#This is working fine
from argopy import DataFetcher
import xarray as xr

# --- 1️⃣ Initialize fetcher with Ifremer ERDDAP ---
fetcher = DataFetcher(src="erddap", server="https://erddap.ifremer.fr/erddap/")

# --- 2️⃣ Define spatial & temporal ranges ---
lon_min, lon_max = 20, 120
lat_min, lat_max = -60, 30
pres_min, pres_max = 0, 2000
time_ranges = [("2023-01-01", "2023-01-31"),
               ("2023-02-01", "2023-02-28"),
               ("2023-03-01", "2023-03-31")]  # add more months if needed

# Split each spatial block into smaller chunks
lon_splits = [(20, 70), (70, 120)]
lat_splits = [(-60, -15), (-15, 30)]

all_floats = []

for lon_block in lon_splits:
    for lat_block in lat_splits:
        for start_date, end_date in time_ranges:
            box = [lon_block[0], lon_block[1], lat_block[0], lat_block[1], pres_min, pres_max, start_date, end_date]
            try:
                df = fetcher.region(box).to_dataframe()
                if not df.empty:
                    all_floats.append(df)
                    print(f"✅ Fetched floats {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
                else:
                    print(f"⚠️ No valid floats in {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
            except Exception as e:
                print(f"❌ Failed {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}: {e}")

# Combine all fetched float data
if all_floats:
    df_all = pd.concat(all_floats).drop_duplicates(subset="platform_number")
    float_ids = df_all['platform_number'].unique()
    print(f"✅ Total unique floats found: {len(float_ids)}")
else:
    float_ids = []
    print("❌ No floats found!")

# --- 3️⃣ Fetch per float and merge ---
datasets = []
for fid in float_ids:
    try:
        ds = fetcher.float(fid).to_xarray()
        datasets.append(ds)
        print(f"✅ Float {fid}: {ds.dims.get('N_PROF', 0)} profiles × {ds.dims.get('N_LEVELS', 0)} levels")
    except Exception as e:
        print(f"❌ Failed to fetch float {fid}: {e}")

# Merge all float datasets into one xarray Dataset
if datasets:
    merged_ds = xr.concat(datasets, dim="N_PROF")
    merged_ds.to_netcdf("merged_argo_data.nc")
    print("✅ All floats merged and saved as merged_argo_data.nc")
else:
    print("❌ No datasets to merge.")



✅ Fetched floats 2023-01-01 → 2023-01-31 | lon 20-70 lat -60--15
✅ Fetched floats 2023-02-01 → 2023-02-28 | lon 20-70 lat -60--15
✅ Fetched floats 2023-03-01 → 2023-03-31 | lon 20-70 lat -60--15
✅ Fetched floats 2023-01-01 → 2023-01-31 | lon 20-70 lat -15-30
✅ Fetched floats 2023-02-01 → 2023-02-28 | lon 20-70 lat -15-30
✅ Fetched floats 2023-03-01 → 2023-03-31 | lon 20-70 lat -15-30
Error: 502, message='Proxy Error', url='https://erddap.ifremer.fr/erddap/tabledap/ArgoFloats.nc?config_mission_number,cycle_number,data_mode,direction,latitude,longitude,platform_number,position_qc,pres,pres_adjusted,pres_adjusted_error,pres_adjusted_qc,pres_qc,psal,psal_adjusted,psal_adjusted_error,psal_adjusted_qc,psal_qc,temp,temp_adjusted,temp_adjusted_error,temp_adjusted_qc,temp_qc,time,time_qc,vertical_sampling_scheme&longitude%3E=70&longitude%3C=120&latitude%3E=-60&latitude%3C=-15&pres%3E=0&pres%3C=2000&time%3E=1672531200.0&time%3C=1675123200.0&latitude!=NaN&longitude!=NaN&distinct()&orderBy(%22time

KeyboardInterrupt: 

In [ ]:
#same like abpve but refined
import pandas as pd
from argopy import DataFetcher
import xarray as xr
from datetime import datetime, timedelta
import os
import time

# --- 1️⃣ Initialize fetcher ---
fetcher = DataFetcher(src="erddap", server="https://erddap.ifremer.fr/erddap/")

# --- 2️⃣ Region & pressure bounds ---
lon_splits = [(20, 70), (70, 120)]
lat_splits = [(-60, -15), (-15, 30)]
pres_min, pres_max = 0, 2000

# --- 3️⃣ Weekly ranges helper ---
def week_ranges(start, end):
    start = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    ranges = []
    while start < end:
        week_end = min(start + timedelta(days=6), end)
        ranges.append((start.strftime("%Y-%m-%d"), week_end.strftime("%Y-%m-%d")))
        start = week_end + timedelta(days=1)
    return ranges

time_ranges = week_ranges("2023-01-01", "2025-06-07")

# --- 4️⃣ Create folder to save individual .nc files ---
os.makedirs("float_nc_files", exist_ok=True)

# --- 5️⃣ Fetch float IDs safely ---
all_floats_df = []

for lon_block in lon_splits:
    for lat_block in lat_splits:
        for start_date, end_date in time_ranges:
            box = [lon_block[0], lon_block[1], lat_block[0], lat_block[1],
                   pres_min, pres_max, start_date, end_date]
            success = False
            for attempt in range(3):
                try:
                    df = fetcher.region(box).to_dataframe()
                    if not df.empty:
                        all_floats_df.append(df)
                        print(f"✅ Fetched {len(df)} rows {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
                    else:
                        print(f"⚠️ No valid floats {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
                    success = True
                    break
                except Exception as e:
                    print(f"❌ Attempt {attempt+1} failed {start_date} → {end_date}: {e}")
                    time.sleep(5)
            if not success:
                print(f"⚠️ Skipped {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")

# Combine all to get unique float IDs
if all_floats_df:
    df_all = pd.concat(all_floats_df, ignore_index=True).drop_duplicates(subset="platform_number")
    float_ids = df_all['platform_number'].unique()
    print(f"\n✅ Total unique floats found: {len(float_ids)}")
else:
    float_ids = []
    print("⚠️ No floats found!")

# --- 6️⃣ Download individual .nc files per float ---
for fid in float_ids:
    try:
        ds = fetcher.float(fid).to_xarray()
        file_path = f"float_nc_files/float_{fid}.nc"
        ds.to_netcdf(file_path)
        print(f"💾 Saved float {fid} → {file_path}")
        del ds  # free memory
    except Exception as e:
        print(f"❌ Failed float {fid}: {e}")

# --- 7️⃣ Merge in chunks to avoid memory overload ---
nc_files = [f"float_nc_files/{f}" for f in os.listdir("float_nc_files") if f.endswith(".nc")]
merged_ds = None
chunk_size = 50  # adjust based on RAM

for i in range(0, len(nc_files), chunk_size):
    chunk_files = nc_files[i:i+chunk_size]
    chunk_ds_list = [xr.open_dataset(f) for f in chunk_files]
    chunk_merged = xr.concat(chunk_ds_list, dim="N_PROF")

    if merged_ds is None:
        merged_ds = chunk_merged
    else:
        merged_ds = xr.concat([merged_ds, chunk_merged], dim="N_PROF")

    # Close opened datasets to free memory
    for ds in chunk_ds_list:
        ds.close()

    print(f"✅ Merged chunk {i//chunk_size + 1}")

# --- 8️⃣ Save final merged dataset ---
if merged_ds is not None:
    merged_ds.to_netcdf("merged_argo_data.nc")
    print("💾 Final merged dataset saved as merged_argo_data.nc")
else:
    print("⚠️ No datasets to merge")


✅ Fetched 50372 rows 2023-01-01 → 2023-01-07 | lon 20-70 lat -60--15
✅ Fetched 51338 rows 2023-01-08 → 2023-01-14 | lon 20-70 lat -60--15
✅ Fetched 61055 rows 2023-01-15 → 2023-01-21 | lon 20-70 lat -60--15
✅ Fetched 55258 rows 2023-01-22 → 2023-01-28 | lon 20-70 lat -60--15
✅ Fetched 52658 rows 2023-01-29 → 2023-02-04 | lon 20-70 lat -60--15
✅ Fetched 58642 rows 2023-02-05 → 2023-02-11 | lon 20-70 lat -60--15
✅ Fetched 57725 rows 2023-02-12 → 2023-02-18 | lon 20-70 lat -60--15
✅ Fetched 55975 rows 2023-02-19 → 2023-02-25 | lon 20-70 lat -60--15
✅ Fetched 54788 rows 2023-02-26 → 2023-03-04 | lon 20-70 lat -60--15
✅ Fetched 65891 rows 2023-03-05 → 2023-03-11 | lon 20-70 lat -60--15
✅ Fetched 59190 rows 2023-03-12 → 2023-03-18 | lon 20-70 lat -60--15
✅ Fetched 53124 rows 2023-03-19 → 2023-03-25 | lon 20-70 lat -60--15
✅ Fetched 54162 rows 2023-03-26 → 2023-04-01 | lon 20-70 lat -60--15
✅ Fetched 60759 rows 2023-04-02 → 2023-04-08 | lon 20-70 lat -60--15
✅ Fetched 54648 rows 2023-04-09 → 

In [ ]:
import pandas as pd
from argopy import DataFetcher
import xarray as xr
from datetime import datetime, timedelta
import time

fetcher = DataFetcher(src="erddap", server="https://erddap.ifremer.fr/erddap/")

lon_splits = [(20, 70), (70, 120)]
lat_splits = [(-60, -15), (-15, 30)]
pres_min, pres_max = 0, 2000

# Function to create weekly ranges
def week_ranges(start, end):
    start = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    ranges = []
    while start < end:
        week_end = min(start + timedelta(days=6), end)
        ranges.append((start.strftime("%Y-%m-%d"), week_end.strftime("%Y-%m-%d")))
        start = week_end + timedelta(days=1)
    return ranges

time_ranges = week_ranges("2023-01-01", "2025-06-07")

all_floats = []

for lon_block in lon_splits:
    for lat_block in lat_splits:
        for start_date, end_date in time_ranges:
            box = [lon_block[0], lon_block[1], lat_block[0], lat_block[1], pres_min, pres_max, start_date, end_date]
            success = False
            for attempt in range(3):  # retry up to 3 times
                try:
                    df = fetcher.region(box).to_dataframe()
                    if not df.empty:
                        all_floats.append(df)
                        print(f"✅ Fetched floats {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
                    else:
                        print(f"⚠️ No valid floats {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")
                    success = True
                    break
                except Exception as e:
                    print(f"❌ Attempt {attempt+1} failed {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}: {e}")
                    time.sleep(5)  # wait before retry
            if not success:
                print(f"⚠️ Skipped {start_date} → {end_date} | lon {lon_block[0]}-{lon_block[1]} lat {lat_block[0]}-{lat_block[1]}")


✅ Fetched floats 2023-01-01 → 2023-01-07 | lon 20-70 lat -60--15
✅ Fetched floats 2023-01-08 → 2023-01-14 | lon 20-70 lat -60--15
✅ Fetched floats 2023-01-15 → 2023-01-21 | lon 20-70 lat -60--15
✅ Fetched floats 2023-01-22 → 2023-01-28 | lon 20-70 lat -60--15
✅ Fetched floats 2023-01-29 → 2023-02-04 | lon 20-70 lat -60--15
✅ Fetched floats 2023-02-05 → 2023-02-11 | lon 20-70 lat -60--15
✅ Fetched floats 2023-02-12 → 2023-02-18 | lon 20-70 lat -60--15
✅ Fetched floats 2023-02-19 → 2023-02-25 | lon 20-70 lat -60--15
✅ Fetched floats 2023-02-26 → 2023-03-04 | lon 20-70 lat -60--15
✅ Fetched floats 2023-03-05 → 2023-03-11 | lon 20-70 lat -60--15
✅ Fetched floats 2023-03-12 → 2023-03-18 | lon 20-70 lat -60--15
✅ Fetched floats 2023-03-19 → 2023-03-25 | lon 20-70 lat -60--15
✅ Fetched floats 2023-03-26 → 2023-04-01 | lon 20-70 lat -60--15
✅ Fetched floats 2023-04-02 → 2023-04-08 | lon 20-70 lat -60--15
✅ Fetched floats 2023-04-09 → 2023-04-15 | lon 20-70 lat -60--15
✅ Fetched floats 2023-04-

KeyboardInterrupt: 

In [ ]:
#Testing Servers
from argopy import DataFetcher
fetcher = DataFetcher(src="erddap", server="https://erddap.ifremer.fr/erddap")
box = [20, 120, -60, 30, 0, 2000, "2023-01-01", "2023-12-31"]   # whole region, 1 year
ds = fetcher.region(box).to_xarray()
print("Ifremer returned profiles:", ds.sizes.get("N_PROF", 0))

In [ ]:
import requests, pandas as pd

API_KEY = "5cb479c1641a9221d30aaa7f82ee354d395baab6"  # put your key here
headers = {"x-argokey": API_KEY}

# Example: fetch profiles for a float/platform
url = "https://argovis-api.colorado.edu/argo/profile"

params = {
    "id": "4902910",   # float/platform ID
    "startDate": "2023-01-01T00:00:00Z",
    "endDate": "2024-01-01T00:00:00Z"
}

r = requests.get(url, headers=headers, params=params, timeout=30)
print("Status:", r.status_code)
print("Preview:", r.text[:500])

if r.ok:
    data = r.json()
    df = pd.json_normalize(data)
    print(df.head())


Status: 404
Preview: {"message":"not found","errors":[{"path":"/argo/profile","message":"not found"}]}
